In [1]:
import transformers
from training_utils import make_supervised_data_module
from collections import namedtuple
from typing import Dict, List

import torch
from tqdm import tqdm

def setup_data_args(
    data_path, base_model, data_length=None, val_split=None, subset=None
):
    DataArgs = namedtuple(
        "DataArgs", ["data_path", "data_length", "val_split", "subset", "is_chat"]
    )
    is_chat = "Llama-3" in base_model and "Instruct" in base_model
    return DataArgs(data_path, data_length, val_split, subset, is_chat)


def load_data(
    data_path: str,
    base_model: str,
    data_length: int = None,
    val_split: float = 0.0,
    subset: str = None,
    model_max_length: int = 1024,  # Set to something large enough to handle the data for eval usecase. Doesnt have to match train.
) -> tuple:
    """
    Load and prepare dataset for training and evaluation.

    Args:
        data_path: Path to the dataset ('meta-math/MetaMathQA' or 'qiaojin/PubMedQA')
        base_model: Name of the base model to use
        data_length: Number of samples to use from dataset
        val_split: Fraction of data to use for validation
        model_max_length: Maximum sequence length for tokenizer. If None, uses defaults (512 for meta-math, 768 for PubMedQA)

    Returns:
        tuple: (train_dataset, eval_dataset, tokenizer)
    """
    # Setup data arguments based on dataset
    if "meta-math" in data_path:
        data_args = setup_data_args(
            data_path, base_model, data_length=data_length, val_split=val_split
        )
    elif data_path == "qiaojin/PubMedQA":
        data_args = setup_data_args(
            data_path,
            base_model,
            data_length=data_length,
            val_split=val_split,
            subset=subset,
        )
    else:
        raise ValueError(f"Unsupported dataset: {data_path}")

    # Initialize tokenizer
    tokenizer = transformers.AutoTokenizer.from_pretrained(
        base_model,
        model_max_length=model_max_length,
        padding_side="right",
        use_fast=False,
    )
    tokenizer.pad_token_id = 2  # unk token, different from eos token

    # Create data module and return datasets
    data_module = make_supervised_data_module(tokenizer=tokenizer, data_args=data_args)
    return data_module, tokenizer

from peft import AutoPeftModelForCausalLM
def load_model(ckpt_path: str) -> AutoPeftModelForCausalLM:
    """
    Load a PEFT model from a checkpoint path.
    
    Args:
        ckpt_path: Path to the model checkpoint
        
    Returns:
        AutoPeftModelForCausalLM: The loaded model
    """
    model = AutoPeftModelForCausalLM.from_pretrained(
        ckpt_path,
        # device_map="auto",
    )
    return model

# Load the model
# ckpt_path = '/root/MoRA/pub-med-qa/save_test_lora_rank128_lr1e-4/checkpoint-1200'
# model = load_model(ckpt_path)

/opt/poetry-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_model_generations(model, prompts: List[str], max_new_tokens: int = 512) -> List[torch.Tensor]:
    """
    Get predictions from the model for a list of prompts.
    
    Args:
        model: The model to get predictions from.
        prompts: List of prompt strings to generate from.
        
    Returns:
        List[torch.Tensor]: The generated outputs from the model.
    """
    predictions = []
    device = model.device
    
    for prompt in tqdm(prompts, desc="Generating.."):
        inputs = tokenizer(prompt, return_tensors="pt")
        input_ids = inputs['input_ids'].to(device)
        attention_mask = inputs['attention_mask'].to(device)
        
        with torch.no_grad():
            prediction = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                pad_token_id=2,
            )

        predictions.append(prediction.cpu().squeeze(0))
            
    return predictions

def get_eval_accuracy(decisions_yhat: List[str], decisions_y: List[str]) -> float:
    """
    Get the evaluation accuracy for a given model and tokenized data.
    """
    assert len(decisions_yhat) == len(decisions_y), f"Lengths are not equal: {len(decisions_yhat)} != {len(decisions_y)}"
    return sum(1 for yhat, y in zip(decisions_yhat, decisions_y) if yhat == y) / len(decisions_yhat)

def get_eval_loss(model, tokenized_data: Dict[str, torch.Tensor], batch_size: int = 8) -> float:
    """
    Get the evaluation loss for a given model and tokenized data.
    
    Args:
        model: The model to get the loss from.
        tokenized_data: The tokenized data to get the loss from.

    Returns:
        float: The average loss.
        List[float]: The loss for each batch.
    """
    loss_by_batch = []
    device = model.device
    num_samples = len(tokenized_data['input_ids'])
    
    for i in tqdm(range(0, num_samples, batch_size), desc="Calculating loss.."):
        batch = {
            k: v[i:i + batch_size].to(device) 
            for k, v in tokenized_data.items()
        }
        
        with torch.no_grad():
            outputs = model(**batch)
            loss_by_batch.append(outputs.loss.mean().item())

    return sum(loss_by_batch) / len(loss_by_batch), loss_by_batch


def extract_decision(label: str) -> str:
    decision = label.split('Final Decision: ')[1].split('<|eot_id|>')[0].strip()
    if decision not in ['yes', 'no', 'maybe']:
        print(f"Invalid decision: {decision}")
        return None
    return decision

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = '/root/MoRA/pub-med-qa/lora_rank128_lr1e-4_witheval/checkpoint-400'
print("Loading model...")
model = load_model(ckpt_path)
model.to(device)

In [3]:
# Load pubmed test set
import json
import uuid


data_path = 'qiaojin/PubMedQA'
subset = 'pqa_labeled'
# subset = 'pqa_artificial'

# Load the data using the new function
data_module, tokenizer = load_data(
    data_path=data_path,
    base_model='meta-llama/Llama-3.2-1B-Instruct',
    subset=subset,
    data_length=1000, # for testing
)

# List[Dict[str, str]] 
# Array with num_samples elements. Each element has two fields, input_ids and labels.
data = data_module['train_dataset'] 
# print(data[0]['labels'])

# Does tokenization and padding
collator = data_module['data_collator']

# Dict[str, torch.Tensor] with keys input_ids, labels, attention_mask
# tokenized_data = collator(data)

# One of ['yes', 'no', 'maybe']
decisions_y = [extract_decision(d['labels']) for d in data] 
decisions_y

# Save to file
with open("pub-med-eval/true_decisions_y.json", "w") as f:
    json.dump(decisions_y, f)

Loading data...


(SOURCES LOG) <|start_header_id|>system<|end_header_id|>

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. <|eot_id|><|start_header_id|>user<|end_header_id|>

### Instruction:
Outcomes of severely injured adult trauma patients in an Australian health service: does trauma center level make a difference?

### Input:
Trauma centers are designated to provide systematized multidisciplinary care to injured patients. Effective trauma systems reduce patient mortality by facilitating the treatment of injured patients at appropriately resourced hospitals. Several U.S. studies report reduced mortality among patients admitted directly to a level I trauma center compared with those admitted to hospitals with less resources. It has yet to be shown whether there is an outcome benefit associated with the "level of hospital" initially treating severely injured trauma patients in Australia. This 

['yes',
 'maybe',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'maybe',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'maybe',
 'no',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'maybe',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'maybe',
 'no',
 'yes',
 'maybe',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'no',
 'no',
 'no',
 'yes',
 'no',
 'yes',
 'maybe',
 'no',
 'no',
 'yes',
 'yes',
 'maybe',
 'yes',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'maybe',
 'yes',
 'yes',
 'no',
 'no',
 'maybe',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'maybe',
 'no',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',


In [5]:
decisions_y

['yes',
 'maybe',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'maybe',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'maybe',
 'no',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'maybe',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'maybe',
 'no',
 'yes',
 'maybe',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'no',
 'no',
 'no',
 'yes',
 'no',
 'yes',
 'maybe',
 'no',
 'no',
 'yes',
 'yes',
 'maybe',
 'yes',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'maybe',
 'yes',
 'yes',
 'no',
 'no',
 'maybe',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'maybe',
 'no',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',


In [ ]:


# Setup input prompts
prompts = [data[i]['input_ids'] for i in range(len(data))]
# print(prompts[0])

# Get model generations
generations = get_model_generations(model, prompts)
generations_str = tokenizer.batch_decode(generations, skip_special_tokens=True)

decisions_yhat = [extract_decision(g) for g in generations_str]
eval_accuracy = get_eval_accuracy(decisions_yhat, decisions_y)

# Save all results
results = {
    'decisions_yhat': decisions_yhat,
    'decisions_y': decisions_y,
    'generations_str': generations_str,
    'eval_accuracy': eval_accuracy,
    'data_path': data_path,
    'subset': subset,
    'ckpt_path': ckpt_path,
}

# Save results to json
id = str(uuid.uuid4())[:8]
with open(f'pub-med-eval/{subset}_results_{id}.json', 'w') as f:
    json.dump(results, f)

In [54]:
# # Labels and Input Ids have same shape. DO NOT USE FOR GENERATION!
# print(tokenizer.decode(tokenized_data['input_ids'][1], skip_special_tokens=True))
# # print(tokenizer.decode(tokenized_data['labels'][1], skip_special_tokens=True)) # Can't be decoded because of -100!
# print(tokenized_data['input_ids'][1].shape, tokenized_data['labels'][1].shape)

# Labels starts with a bunch of -100s which is the IGNORE_INDEX
# because the input ids locations are not considered part of the loss
# The loss is only computed for the labels that are not -100, which are in the middle.
# Finally there is a bunch of -100s at the end for padding but ignored for loss.

# Input ids meanwhile starts with the token ids (no left ignore) and ends with padding tokens (2).

In [57]:
# E2E generation example from string -> tokenized -> model -> decoded output
# test_input = tokenizer.encode("Hello, how are you?", return_tensors="pt").to(device)
# print(tokenizer.batch_decode(model.generate(test_input, max_new_tokens=128).cpu(), pad_token_id=2))

In [ ]:
# Loss values are not matching the training log loss values
# eval_loss, eval_loss_by_batch = get_eval_loss(model, tokenized_data)
# print(eval_loss)